In [1]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [2]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def ranktoset (A):
    A = list(A)
    sets = [[A[0]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append(new)
    return(sets)


In [11]:
def robustcheck(a,R,r,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    print(prob.value - (1-np.sum(a))*r_f)
    return(prob.value,q.value,q_b.value)

In [20]:
def dual (sets,p,R,r,m,r_f,c):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable(M, nonneg= True)
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    w = cp.Variable(N)
    z = cp.Variable(M)
    s = cp.Variable(N)
    z1 = 0
    constraints = [w - gamma*(np.zeros(N)+1) <= t]
    for i in range(N):
        lbdasum = 0
        vsum = 0
        for j in range(M):
            if i in sets[j]:
                lbdasum = lbdasum + lbda[j]
                vsum = vsum + v[j]
        constraints.append((-R @ a)[i] - lbdasum - beta - (1-cp.sum(a))*r_f <= 0)
        constraints.append(s[i] == -alpha + vsum)
        constraints.append(cp.kl_div(gamma,w[i])+gamma+s[i]-w[i]<= 0)
        z1 = z1 + p[i]*t[i]
    for j in range(M):
        constraints.append(cp.pos(-(1-m)*v[j]+lbda[j]) <= z[j])
    constraints.append(cp.abs(a)<= 10)
    constraints.append(alpha + beta + gamma * r  + cp.sum(z) + p@t <= c)
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)

In [5]:
np.random.seed(10)

In [6]:
N=6
p = (np.zeros(N)+1)*1/N
I = 2
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))


[0.08701171 0.08745984]


In [18]:
a = np.zeros(I)+1/I
r = 0.043
m = 0.95
r_f = 0.001
c = 0.3
robustcheck(a,R,r,p,m,r_f)

0.10537841418910004


(0.10537841418910004,
 array([0.1790008 , 0.15890488, 0.16356101, 0.16856318, 0.15966888,
        0.17030125]),
 array([1.21607421e-11, 1.00000000e+00, 3.36866518e-11, 2.32092980e-11,
        8.06106204e-12, 2.88405403e-14]))

In [14]:
x=np.arange(1,N)
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]

In [21]:
sets = psets
dual (sets,p,R,r,m,r_f,c)

(array([2.59466842, 7.89932368]), 0.9071461509844918)

In [54]:
[probv,vv,lbdav,alphav,betav,gammav,tv]=dual (sets,p,R,r,m,r_f,a)
N = len(p)
M = len(sets)
cons1 =np.zeros(N)
cons2 = np.zeros(N)
z0 = 0
for j in range(M):
    z9 = -np.min(vv[j,sets[j]])*(1-m)+lbdav[j]
    z0 = z0 + max(z9,0)
for i in range(N):
    lbdsom = 0
    for j in range(M):
        if i in sets[j]:
            lbdsom = lbdsom + lbdav[j]
    cons1[i] = R.dot(a)[i] + betav + lbdsom
    cons2[i] = gammav * np.exp((-alphav+sum(vv[0:M:1,i]))/gammav)-tv[i]
print(cons1)
print(cons2)
print(-1+alphav+betav+gammav*r+sum(p*tv)+z0)

[ 5.70003067e-09 -5.38403810e-09  6.53539289e+00  1.70622106e+01]
[-8.10329546e-08 -2.98001260e-06 -3.94966149e-08 -2.06583066e-08]
[24.65265508]
